In [ ]:
import os, glob
import numpy as np
import tensorflow as tf
import pandas as pd

## Prepare dataset

In [ ]:
CAPTURES = "../dataset/captures"
MAX_LEN  = 100 
N_CH     = 6 
classes = sorted(d for d in os.listdir(CAPTURES)
                 if os.path.isdir(os.path.join(CAPTURES, d)))

class_to_id = {c: i for i, c in enumerate(classes)}

lengths = []
"""
Load and preprocess the gesture recordings to create the dataset for training the MLP.

For each gesture class, the function:
1. Loads the accelerometer and gyroscope measurements from CSV files.
2. Creates a fixed-size sequence using zero-padding or truncation.
3. Flattens the 2D sensor matrix into a 1D feature vector.
4. Assigns a numerical label to identify the gesture.

Finally, X contains the input features and y contains the corresponding gesture labels.
"""

X, y = [], []
for cls in classes:
    for path in glob.glob(os.path.join(CAPTURES, cls, "*.csv")):
        rows = np.loadtxt(path, delimiter=",", skiprows=1, usecols=range(4, 10))
        seq = np.zeros((MAX_LEN, N_CH), dtype=np.float32)   
        n = min(rows.shape[0], MAX_LEN)
        seq[:n] = rows[:n]
        X.append(seq.flatten())          
        y.append(class_to_id[cls])

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int32)
print("X", X.shape, "classes", len(classes))  

## Analyze dataset

In [ ]:
print(X.shape)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)   # stratify: 16 classes

scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train).astype(np.float32)
X_test  = scaler.transform(X_test).astype(np.float32)


model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(N_CH * MAX_LEN,)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(64,  activation="relu"),
    tf.keras.layers.Dense(len(classes), activation="softmax"),
])
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

model.fit(X_train, y_train, batch_size=32, epochs=100, validation_data=(X_test, y_test))

## Test trained network

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

loss, acc = model.evaluate(X_test, y_test, verbose=0)
y_pred = model.predict(X_test, verbose=0).argmax(1)

print(f"test loss {loss:.4f}  accuracy {acc:.4f}  ({len(y_test)} samples)")
print(classification_report(y_test, y_pred, target_names=classes, digits=3, zero_division=0))

cm = confusion_matrix(y_test, y_pred)
for i, c in enumerate(classes):
    print(f"{c:28s} acc={cm[i, i] / cm[i].sum():.2f}  n={cm[i].sum()}")

## Architecture grid search (size vs accuracy)

The first Dense layer holds 76.8k of the 86.2k parameters, so input width dominates the size. Grid over temporal average-pooling (`stride` frames -> one) and hidden layer sizes; every model gets the same epochs and the same split. `int8_kb` assumes 1 byte per parameter (the real TFLite size is measured in the next cell).

99 test samples means roughly +/- 3 points standard error, so treat differences under ~5 points as noise.

In [ ]:
GRID_STRIDES = [1, 2, 4]                      # average this many frames together
GRID_HIDDEN  = [(128, 64), (64, 32), (32, 16), (16,)]
GRID_EPOCHS  = 80


def fold(Xs, stride):
    """Average-pool groups of `stride` frames; MAX_LEN is divisible by 1/2/4."""
    if stride == 1:
        return Xs
    return Xs.reshape(-1, MAX_LEN // stride, stride, N_CH).mean(2).reshape(len(Xs), -1)


rows, models, splits = [], {}, {}
for stride in GRID_STRIDES:
    Xtr, Xte = fold(X_train, stride), fold(X_test, stride)
    for hidden in GRID_HIDDEN:
        name = "x".join(map(str, hidden))
        m = tf.keras.Sequential(
            [tf.keras.layers.Input((Xtr.shape[1],))]
            + [tf.keras.layers.Dense(h, activation="relu") for h in hidden]
            + [tf.keras.layers.Dense(len(classes), activation="softmax")])
        m.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
        m.fit(Xtr, y_train, batch_size=32, epochs=GRID_EPOCHS, verbose=0)
        acc = m.evaluate(Xte, y_test, verbose=0)[1]
        models[(stride, name)], splits[(stride, name)] = m, (Xtr, Xte)
        rows.append(dict(stride=stride, hidden=name, params=m.count_params(),
                         int8_kb=m.count_params() / 1024, test_acc=round(acc, 4)))

grid = pd.DataFrame(rows).sort_values("params").reset_index(drop=True)
print(grid.to_string(index=False, float_format="{:.3f}".format))

# smallest model within 2 points of the best accuracy on this grid
best_acc = grid.test_acc.max()
pick = grid[grid.test_acc >= best_acc - 0.02].iloc[0]
best_model = models[(pick.stride, pick.hidden)]
best_X_train, best_X_test = splits[(pick.stride, pick.hidden)]
best_model.save("model_best.keras")
print(f"\npicked stride={pick.stride} hidden={pick.hidden}: "
      f"~{pick['int8_kb']:.1f}KB int8, test acc {pick.test_acc:.4f} "
      f"(best on grid {best_acc:.4f})")

## TensorFlow Lite conversion (ESP32)

Float32 TFLite for reference, and full int8 quantization (weights + activations) which is what an ESP32 / TFLite Micro should run. The grid winner is converted with its own (pooled) representative data.

The scaler lives outside the model, so its parameters are saved separately for the firmware.

In [ ]:
def to_tflite(src, int8, rep):
    conv = tf.lite.TFLiteConverter.from_keras_model(src)
    if int8:
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
        conv.representative_dataset = lambda: ([x[None]] for x in rep)
        conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        conv.inference_input_type = tf.int8
        conv.inference_output_type = tf.int8
    return conv.convert()


def tflite_accuracy(blob, X, y):
    it = tf.lite.Interpreter(model_content=blob)
    it.allocate_tensors()
    inp, out = it.get_input_details()[0], it.get_output_details()[0]
    hits = 0
    for x, label in zip(X, y):
        if inp["dtype"] == np.int8:                      # quantize the input like the MCU will
            s, z = inp["quantization"]
            x = np.clip(np.round(x / s + z), -128, 127).astype(np.int8)
        it.set_tensor(inp["index"], x[None].astype(inp["dtype"]))
        it.invoke()
        hits += int(it.get_tensor(out["index"])[0].argmax() == label)  # scale > 0, so argmax survives
    return hits / len(y)


exports = {
    "model_float32.tflite": (to_tflite(model, False, X_train), X_test),
    "model_int8.tflite": (to_tflite(model, True, X_train), X_test),
    "model_best_int8.tflite": (to_tflite(best_model, True, best_X_train), best_X_test),
}
for name, (blob, Xte) in exports.items():
    open(name, "wb").write(blob)
    print(f"{name:26s} {len(blob) / 1024:6.1f}KB  test acc {tflite_accuracy(blob, Xte, y_test):.4f}")

# firmware must normalise raw sensor rows with the same scaler before inference
np.savez("scaler.npz", mean=scaler.mean_, scale=scaler.scale_)